# Notebook 03 — Empirical Analysis

Phase 3 of the "Stablecoins vs. SWIFT" project. Tests four
hypotheses (H1–H4) against the master datasets built in Phase 2B.

**Structure:**
- §0 Setup: imports, seed, style, data loads, smoke test
- §1 H1 Metcalfe's Law (log-log OLS, ADF, Wald, structural break)
- §2 H3 Market concentration (HHI trend, event decomposition)
- §3 H4 Cost friction (paired tests at $200 and $10,000)
- §4 H2 Diffusion (pooled → country FE → two-way FE ladder)

All methodology decisions are logged in
`docs/PHASE_3_DECISIONS.md` (D-01 through D-10). Each section
below references the relevant decision IDs.

**Execution order rationale:** H1 opens the notebook because
Metcalfe's Law is the foundational network-effects claim that
the rest of the analysis builds on. H2 closes the notebook
because its specification ladder is the most complex.

## §0 — Setup

Imports, random seed, matplotlib style, data loads, assertions,
linearmodels smoke test. All setup lives in §0; no imports or
seeds later in the notebook.

In [1]:
# Standard library
import random
from pathlib import Path

# Numerical and data
import numpy as np
import pandas as pd

# Econometrics
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from linearmodels.panel import PanelOLS

# Stats
from scipy import stats

# Plotting
import matplotlib.pyplot as plt
import matplotlib as mpl

In [2]:
# D-06: global random seed
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [3]:
# D-05: figure style conventions
mpl.rcParams["figure.dpi"] = 100         # screen display
mpl.rcParams["savefig.dpi"] = 300        # saved output
mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["axes.grid"] = False
mpl.rcParams["axes.spines.top"] = False
mpl.rcParams["axes.spines.right"] = False

# Three-color palette for all Phase 3 figures
PALETTE = {
    "primary":   "#1f77b4",   # tab:blue
    "secondary": "#ff7f0e",   # tab:orange
    "tertiary":  "#2ca02c",   # tab:green
    "muted":     "#7f7f7f",   # tab:gray (for annotations)
}

In [4]:
# D-08: no hardcoded paths
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_DIR = REPO_ROOT / "data" / "03_processed"
FIG_DIR = REPO_ROOT / "outputs" / "figures"
TBL_DIR = REPO_ROOT / "outputs" / "tables"

assert DATA_DIR.exists(), f"DATA_DIR missing: {DATA_DIR}"
assert FIG_DIR.exists(), f"FIG_DIR missing: {FIG_DIR}"
assert TBL_DIR.exists(), f"TBL_DIR missing: {TBL_DIR}"
print(f"REPO_ROOT: {REPO_ROOT}")

REPO_ROOT: C:\dev\ine


In [5]:
# Load all four master datasets with expected shapes (from
# Phase 2C Task A verification).
h1 = pd.read_csv(DATA_DIR / "h1_network_effects.csv",
                 parse_dates=["date"])
h2 = pd.read_csv(DATA_DIR / "h2_diffusion_dataset.csv")
h3 = pd.read_csv(DATA_DIR / "h3_concentration.csv",
                 parse_dates=["date"])
h4 = pd.read_csv(DATA_DIR / "h4_infrastructure_cost.csv")

# Shape assertions match Phase 2C baseline exactly
assert h1.shape == (4384, 4), f"h1 shape mismatch: {h1.shape}"
assert h2.shape == (861, 23),  f"h2 shape mismatch: {h2.shape}"
assert h3.shape == (72, 7),    f"h3 shape mismatch: {h3.shape}"
assert h4.shape == (72, 12),   f"h4 shape mismatch: {h4.shape}"

# Content assertions
assert set(h1["asset"].unique()) == {"USDC", "USDT"}
assert h1["date"].min() == pd.Timestamp("2020-01-01")
assert h1["date"].max() == pd.Timestamp("2025-12-31")
assert h3["date"].min() == pd.Timestamp("2020-01-01")
assert h3["date"].max() == pd.Timestamp("2025-12-01")
assert h2["year"].min() == 2020
assert h2["year"].max() == 2025
assert h4["month"].min() == "2020-01"
assert h4["month"].max() == "2025-12"

print("All four master datasets loaded.")
print(f"  h1: {h1.shape}  assets: {sorted(h1['asset'].unique())}")
print(f"  h2: {h2.shape}  countries: {h2['country_iso3'].nunique()}")
print(f"  h3: {h3.shape}  months: {len(h3)}")
print(f"  h4: {h4.shape}  months: {len(h4)}")

All four master datasets loaded.
  h1: (4384, 4)  assets: ['USDC', 'USDT']
  h2: (861, 23)  countries: 160
  h3: (72, 7)  months: 72
  h4: (72, 12)  months: 72


### §0.1 Smoke test — linearmodels compatibility

Day-1 infrastructure check. Verifies `linearmodels==7.0` fits a
`PanelOLS` model against the current `pandas` / `numpy` versions.
If this fails, STOP and escalate before doing any H2 analysis
work. This is not a real regression — it's a handshake with the
dependency.

In [6]:
# Smoke test: PanelOLS must return non-null parameters on a
# trivial fit. Uses h2's MultiIndex (country_iso3, year).
_smoke = h2.dropna(subset=["gdp_per_capita_usd"]).set_index(
    ["country_iso3", "year"]
)
_smoke_fit = PanelOLS(
    _smoke["adoption_percentile"],
    sm.add_constant(_smoke[["gdp_per_capita_usd"]]),
    entity_effects=True,
).fit()
assert _smoke_fit.params.notna().all(), \
    "linearmodels smoke test failed — PanelOLS returned NaN params"
print("linearmodels smoke test: OK")
print(f"  observations: {_smoke_fit.nobs}")
print(f"  entities: {_smoke_fit.entity_info.total}")
del _smoke, _smoke_fit

linearmodels smoke test: OK
  observations: 859
  entities: 160.0


## §1 — H1 Metcalfe's Law

Tests whether stablecoin transfer activity scales with active
addresses in log-log space. Primary spec: log-log OLS with
Newey-West HAC standard errors, per asset. Wald tests at β = 1
(linear) and β = 2 (strict Metcalfe). ADF stationarity pre-check.
Pre/post structural break at 2022-11-11 (FTX).

**Decisions invoked:** D-03 (structural break date),
D-06 (seed), D-09 (HAC maxlags = 12 for daily data).

**Subsections (to be implemented in Prompt 3):**
- §1.1 Log transforms and positivity checks
- §1.2 ADF stationarity tests (levels and first differences)
- §1.3 Log-log OLS per asset, full window
- §1.4 Wald tests: H0: β=1, H0: β=2
- §1.5 Pre/post structural break sub-samples

### §1.1 Log transforms and positivity checks

Both H1 variables (`transfer_count`, `active_addresses`) enter
the regression in logs. Before log-transforming, assert strict
positivity per asset. Any zero-valued days would produce `-inf`
and must be surfaced, not silently filtered.

In [7]:
# §1.1 — Positivity checks before log transform
for asset in ["USDC", "USDT"]:
    sub = h1[h1["asset"] == asset]
    n_zero_tc = (sub["transfer_count"] <= 0).sum()
    n_zero_aa = (sub["active_addresses"] <= 0).sum()
    assert n_zero_tc == 0, \
        f"{asset}: {n_zero_tc} non-positive transfer_count rows"
    assert n_zero_aa == 0, \
        f"{asset}: {n_zero_aa} non-positive active_addresses rows"

# Log transforms (D-11 derived variables in DATA_DICTIONARY.md)
h1 = h1.sort_values(["asset", "date"]).reset_index(drop=True)
h1["log_transfer_count"] = np.log(h1["transfer_count"])
h1["log_active_addresses"] = np.log(h1["active_addresses"])

assert h1["log_transfer_count"].notna().all()
assert h1["log_active_addresses"].notna().all()
assert np.isfinite(h1["log_transfer_count"]).all()
assert np.isfinite(h1["log_active_addresses"]).all()

print("Positivity checks passed. Log transforms applied.")
print(h1[["asset", "date", "log_transfer_count",
          "log_active_addresses"]].head(3).to_string(index=False))

Positivity checks passed. Log transforms applied.
asset       date  log_transfer_count  log_active_addresses
 USDC 2020-01-01            7.731931              7.300473
 USDC 2020-01-02            8.077447              7.741534
 USDC 2020-01-03            8.328934              7.848153


### §1.2 ADF stationarity pre-check

Augmented Dickey-Fuller tests on both variables, both assets,
in levels and first differences. Per D-12, ADF is a pre-check
(not a gate) — results inform whether §1.3 reports a first-
differenced robustness row.

In [8]:
# §1.2 — ADF stationarity tests
# Note: adfuller is imported in Cell 3 (§0 imports). We rely on
# that top-level import per D-08 (all imports in cell 1 only).

adf_rows = []
for asset in ["USDC", "USDT"]:
    sub = h1[h1["asset"] == asset].sort_values("date")
    for var in ["log_transfer_count", "log_active_addresses"]:
        # Levels
        stat, pval, _, nobs, crit, _ = adfuller(
            sub[var], regression="c", autolag="AIC")
        adf_rows.append({
            "asset": asset, "variable": var,
            "specification": "levels",
            "adf_stat": stat, "p_value": pval,
            "n_obs": nobs, "crit_5pct": crit["5%"],
            "stationary_at_5pct": pval < 0.05,
        })
        # First differences
        diff = sub[var].diff().dropna()
        stat, pval, _, nobs, crit, _ = adfuller(
            diff, regression="c", autolag="AIC")
        adf_rows.append({
            "asset": asset, "variable": var,
            "specification": "first_difference",
            "adf_stat": stat, "p_value": pval,
            "n_obs": nobs, "crit_5pct": crit["5%"],
            "stationary_at_5pct": pval < 0.05,
        })

adf_df = pd.DataFrame(adf_rows)
adf_df.to_csv(TBL_DIR / "tbl_h1_adf_tests.csv", index=False)
with open(TBL_DIR / "tbl_h1_adf_tests.tex", "w") as f:
    f.write(adf_df.to_latex(index=False, float_format="%.4f"))

assert (TBL_DIR / "tbl_h1_adf_tests.csv").exists()
assert (TBL_DIR / "tbl_h1_adf_tests.tex").exists()

print("ADF test results:")
print(adf_df.to_string(index=False))

# Flag any non-stationary levels for §1.3 robustness
non_stationary_levels = adf_df[
    (adf_df["specification"] == "levels")
    & (~adf_df["stationary_at_5pct"])
]
if len(non_stationary_levels) > 0:
    print("\nD-12 triggered: the following series are non-"
          "stationary in levels. §1.3 will include first-"
          "differenced robustness regressions.")
    print(non_stationary_levels[
        ["asset", "variable", "p_value"]].to_string(index=False))
else:
    print("\nAll series stationary in levels. No first-differenced "
          "robustness needed per D-12.")

ADF test results:
asset             variable    specification   adf_stat      p_value  n_obs  crit_5pct  stationary_at_5pct
 USDC   log_transfer_count           levels  -2.165092 2.191581e-01   2169  -2.862873               False
 USDC   log_transfer_count first_difference -11.575237 3.033890e-21   2164  -2.862877                True
 USDC log_active_addresses           levels  -1.883188 3.400106e-01   2165  -2.862876               False
 USDC log_active_addresses first_difference -10.409333 1.821414e-18   2164  -2.862877                True
 USDT   log_transfer_count           levels  -4.352537 3.598180e-04   2165  -2.862876                True
 USDT   log_transfer_count first_difference -13.920857 5.298962e-26   2164  -2.862877                True
 USDT log_active_addresses           levels  -4.255853 5.292528e-04   2165  -2.862876                True
 USDT log_active_addresses first_difference -12.902098 4.227013e-24   2164  -2.862877                True

D-12 triggered: the followi

### §1.3 Log-log OLS per asset, full window

Primary specification: `log_transfer_count ~ log_active_addresses`
per asset, with Newey-West HAC standard errors (maxlags=12 per
D-09 for n ≈ 2,192 daily observations).

Per D-12, if any variable was flagged non-stationary in levels
in §1.2, a first-differenced robustness regression is added
alongside the levels result.

In [9]:
# §1.3 — Log-log OLS per asset, full window, with HAC SE
HAC_MAXLAGS_DAILY = 12  # D-09: Newey (1994) rule of thumb
                         # for n ≈ 2,192

ols_rows = []
ols_results = {}  # keep fitted objects for §1.4 Wald tests

for asset in ["USDC", "USDT"]:
    sub = h1[h1["asset"] == asset].sort_values("date")

    # Levels (headline)
    y = sub["log_transfer_count"].values
    X = sm.add_constant(sub["log_active_addresses"].values)
    res = sm.OLS(y, X).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": HAC_MAXLAGS_DAILY},
    )
    ols_results[(asset, "levels")] = res

    beta = res.params[1]
    beta_se = res.bse[1]
    ci_low, ci_high = res.conf_int(alpha=0.05)[1]
    ols_rows.append({
        "asset": asset,
        "specification": "levels",
        "n_obs": int(res.nobs),
        "alpha": res.params[0],
        "beta": beta,
        "beta_se": beta_se,
        "beta_ci_low": ci_low,
        "beta_ci_high": ci_high,
        "r_squared": res.rsquared,
        "hac_maxlags": HAC_MAXLAGS_DAILY,
    })

    # First-differenced robustness (if D-12 triggered)
    if len(non_stationary_levels) > 0:
        y_diff = sub["log_transfer_count"].diff().dropna().values
        x_diff = sub["log_active_addresses"].diff().dropna().values
        X_diff = sm.add_constant(x_diff)
        res_diff = sm.OLS(y_diff, X_diff).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": HAC_MAXLAGS_DAILY},
        )
        ols_results[(asset, "first_diff")] = res_diff

        beta_d = res_diff.params[1]
        beta_se_d = res_diff.bse[1]
        ci_low_d, ci_high_d = res_diff.conf_int(alpha=0.05)[1]
        ols_rows.append({
            "asset": asset,
            "specification": "first_difference",
            "n_obs": int(res_diff.nobs),
            "alpha": res_diff.params[0],
            "beta": beta_d,
            "beta_se": beta_se_d,
            "beta_ci_low": ci_low_d,
            "beta_ci_high": ci_high_d,
            "r_squared": res_diff.rsquared,
            "hac_maxlags": HAC_MAXLAGS_DAILY,
        })

ols_df = pd.DataFrame(ols_rows)
ols_df.to_csv(TBL_DIR / "tbl_h1_ols_fullwindow.csv", index=False)
with open(TBL_DIR / "tbl_h1_ols_fullwindow.tex", "w") as f:
    f.write(ols_df.to_latex(index=False, float_format="%.4f"))

# Archive full .summary() per D-08 rule 7
with open(TBL_DIR / "tbl_h1_ols_fullwindow_summary.txt", "w") as f:
    for (asset, spec), res in ols_results.items():
        f.write(f"=== {asset} — {spec} ===\n")
        f.write(res.summary().as_text())
        f.write("\n\n")

assert (TBL_DIR / "tbl_h1_ols_fullwindow.csv").exists()
assert (TBL_DIR / "tbl_h1_ols_fullwindow.tex").exists()
assert (TBL_DIR / "tbl_h1_ols_fullwindow_summary.txt").exists()

print("H1 full-window OLS results:")
print(ols_df.to_string(index=False))

H1 full-window OLS results:
asset    specification  n_obs    alpha     beta  beta_se  beta_ci_low  beta_ci_high  r_squared  hac_maxlags
 USDC           levels   2192 1.524823 0.982066 0.028958     0.925309      1.038824   0.874093           12
 USDC first_difference   2191 0.001593 0.702798 0.063285     0.578762      0.826835   0.442986           12
 USDT           levels   2192 0.509937 1.014475 0.007290     1.000186      1.028764   0.986653           12
 USDT first_difference   2191 0.000272 0.926920 0.023031     0.881780      0.972059   0.840811           12


### §1.4 Wald tests: H0: β=1 and H0: β=2

Tests whether the elasticity of transfers with respect to
addresses is consistent with linear scaling (β=1) or strict
Metcalfe (β=2). For count-based DV on stablecoins, literature
expects 1 < β < 2 (super-linear but sub-Metcalfe).

In [10]:
# §1.4 — Wald tests on the levels regression per asset
wald_rows = []
for asset in ["USDC", "USDT"]:
    res = ols_results[(asset, "levels")]
    # H0: β = 1
    w1 = res.wald_test("x1 = 1", scalar=True)
    # H0: β = 2
    w2 = res.wald_test("x1 = 2", scalar=True)
    wald_rows.append({
        "asset": asset,
        "beta": res.params[1],
        "wald_stat_beta1": float(w1.statistic),
        "p_value_beta1": float(w1.pvalue),
        "reject_beta1_at_5pct": float(w1.pvalue) < 0.05,
        "wald_stat_beta2": float(w2.statistic),
        "p_value_beta2": float(w2.pvalue),
        "reject_beta2_at_5pct": float(w2.pvalue) < 0.05,
    })

wald_df = pd.DataFrame(wald_rows)
wald_df.to_csv(TBL_DIR / "tbl_h1_wald_tests.csv", index=False)
with open(TBL_DIR / "tbl_h1_wald_tests.tex", "w") as f:
    f.write(wald_df.to_latex(index=False, float_format="%.4f"))

assert (TBL_DIR / "tbl_h1_wald_tests.csv").exists()

print("H1 Wald tests:")
print(wald_df.to_string(index=False))

H1 Wald tests:
asset     beta  wald_stat_beta1  p_value_beta1  reject_beta1_at_5pct  wald_stat_beta2  p_value_beta2  reject_beta2_at_5pct
 USDC 0.982066         0.383515       0.535728                 False      1235.629268  1.101942e-270                  True
 USDT 1.014475         3.942138       0.047091                  True     18273.991064   0.000000e+00                  True


### §1.4b Engle-Granger cointegration test

Tests whether the levels regression residuals are stationary. If
residuals are stationary (p < 0.05 on ADF), the two series
cointegrate and the levels β is a valid long-run elasticity
estimate. If non-stationary, the levels regression is spurious
and the first-differenced β is the preferred headline. Per D-13.

In [11]:
# §1.4b — Engle-Granger cointegration test
cointegration_rows = []
for asset in ["USDC", "USDT"]:
    res = ols_results[(asset, "levels")]
    residuals = res.resid

    # ADF on residuals, no constant (standard Engle-Granger specification)
    eg_stat, eg_pval, _, eg_nobs, eg_crit, _ = adfuller(
        residuals, regression="n", autolag="AIC")
    cointegration_rows.append({
        "asset": asset,
        "adf_stat_residuals": eg_stat,
        "p_value": eg_pval,
        "n_obs": eg_nobs,
        "crit_5pct": eg_crit["5%"],
        "cointegrated_at_5pct": eg_pval < 0.05,
    })

coint_df = pd.DataFrame(cointegration_rows)
coint_df.to_csv(TBL_DIR / "tbl_h1_cointegration.csv", index=False)
with open(TBL_DIR / "tbl_h1_cointegration.tex", "w") as f:
    f.write(coint_df.to_latex(index=False, float_format="%.4f"))

assert (TBL_DIR / "tbl_h1_cointegration.csv").exists()

print("Engle-Granger cointegration test (on levels residuals):")
print(coint_df.to_string(index=False))
for row in cointegration_rows:
    verdict = ("COINTEGRATED — levels β valid"
               if row["cointegrated_at_5pct"]
               else "NOT COINTEGRATED — first-diff β preferred")
    print(f"  {row['asset']}: {verdict} (p={row['p_value']:.4f})")

Engle-Granger cointegration test (on levels residuals):
asset  adf_stat_residuals  p_value  n_obs  crit_5pct  cointegrated_at_5pct
 USDC           -2.441117 0.014133   2169  -1.941125                  True
 USDT           -4.823009 0.000002   2178  -1.941124                  True
  USDC: COINTEGRATED — levels β valid (p=0.0141)
  USDT: COINTEGRATED — levels β valid (p=0.0000)


### §1.4c Cook's distance — influence diagnostic

Computes Cook's distance for every observation in each asset's
levels regression. Observations with Cook's D > 4/n are flagged
as influential per Cook (1977). Reports both the full regression
and a robustness regression excluding influential points, per
D-14.

In [12]:
# §1.4c — Cook's distance and robustness regression
from statsmodels.stats.outliers_influence import OLSInfluence

cooks_rows = []
cooks_details = {}  # store per-asset arrays for §1.4d

for asset in ["USDC", "USDT"]:
    sub = h1[h1["asset"] == asset].sort_values("date").reset_index(
        drop=True)
    y = sub["log_transfer_count"].values
    X = sm.add_constant(sub["log_active_addresses"].values)
    # Refit WITHOUT HAC for influence diagnostic — Cook's D is
    # computed on the standard OLS leverage, not HAC residuals
    res_ols = sm.OLS(y, X).fit()
    infl = OLSInfluence(res_ols)
    cooks_d = infl.cooks_distance[0]
    threshold = 4.0 / len(cooks_d)
    influential_mask = cooks_d > threshold
    n_influential = int(influential_mask.sum())

    # Refit with HAC on the NON-influential subset
    y_clean = y[~influential_mask]
    X_clean = X[~influential_mask]
    res_clean = sm.OLS(y_clean, X_clean).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": HAC_MAXLAGS_DAILY},
    )

    cooks_rows.append({
        "asset": asset,
        "n_total": len(cooks_d),
        "threshold_4_over_n": threshold,
        "n_influential": n_influential,
        "pct_influential": 100.0 * n_influential / len(cooks_d),
        "beta_full": ols_results[(asset, "levels")].params[1],
        "beta_ex_influential": res_clean.params[1],
        "beta_shift": res_clean.params[1]
                      - ols_results[(asset, "levels")].params[1],
        "r2_full": ols_results[(asset, "levels")].rsquared,
        "r2_ex_influential": res_clean.rsquared,
    })

    cooks_details[asset] = {
        "cooks_d": cooks_d,
        "dates": sub["date"].values,
        "log_aa": sub["log_active_addresses"].values,
        "log_tc": sub["log_transfer_count"].values,
        "active_addresses": sub["active_addresses"].values,
        "transfer_count": sub["transfer_count"].values,
        "threshold": threshold,
    }

cooks_df = pd.DataFrame(cooks_rows)
cooks_df.to_csv(TBL_DIR / "tbl_h1_cooks_influence.csv", index=False)
with open(TBL_DIR / "tbl_h1_cooks_influence.tex", "w") as f:
    f.write(cooks_df.to_latex(index=False, float_format="%.4f"))

assert (TBL_DIR / "tbl_h1_cooks_influence.csv").exists()

print("Cook's distance influence analysis:")
print(cooks_df.to_string(index=False))
for row in cooks_rows:
    print(f"\n  {row['asset']}: "
          f"{row['n_influential']} influential obs "
          f"({row['pct_influential']:.1f}% of sample). "
          f"β shifts from {row['beta_full']:.4f} to "
          f"{row['beta_ex_influential']:.4f} "
          f"(Δ = {row['beta_shift']:+.4f}).")

Cook's distance influence analysis:
asset  n_total  threshold_4_over_n  n_influential  pct_influential  beta_full  beta_ex_influential  beta_shift  r2_full  r2_ex_influential
 USDC     2192            0.001825            164         7.481752   0.982066             1.042628    0.060562 0.874093           0.931550
 USDT     2192            0.001825            102         4.653285   1.014475             1.019987    0.005512 0.986653           0.994402

  USDC: 164 influential obs (7.5% of sample). β shifts from 0.9821 to 1.0426 (Δ = +0.0606).

  USDT: 102 influential obs (4.7% of sample). β shifts from 1.0145 to 1.0200 (Δ = +0.0055).


### §1.4d Top influential USDC observations by date

Lists the 10 observations with the highest Cook's distance for
USDC, with their date, active_addresses, transfer_count, and
Cook's D value. This lets Phase 4 narrative attribute specific
dates to specific economic events (e.g., DeFi bot spikes in
early 2020, SVB depeg in March 2023). Per D-14.

In [13]:
# §1.4d — Top 10 highest Cook's D USDC observations
det = cooks_details["USDC"]
top10_idx = np.argsort(det["cooks_d"])[::-1][:10]
top10_df = pd.DataFrame({
    "date": pd.to_datetime(det["dates"][top10_idx]).strftime(
        "%Y-%m-%d"),
    "active_addresses": det["active_addresses"][top10_idx].astype(int),
    "transfer_count": det["transfer_count"][top10_idx].astype(int),
    "log_active_addresses": np.round(det["log_aa"][top10_idx], 3),
    "log_transfer_count": np.round(det["log_tc"][top10_idx], 3),
    "cooks_d": np.round(det["cooks_d"][top10_idx], 4),
    "above_threshold": det["cooks_d"][top10_idx] > det["threshold"],
})

top10_df.to_csv(TBL_DIR / "tbl_h1_usdc_top10_influential.csv",
                index=False)
with open(TBL_DIR / "tbl_h1_usdc_top10_influential.tex", "w") as f:
    f.write(top10_df.to_latex(index=False))

assert (TBL_DIR / "tbl_h1_usdc_top10_influential.csv").exists()

print("Top 10 most influential USDC observations (by Cook's D):")
print(top10_df.to_string(index=False))

Top 10 most influential USDC observations (by Cook's D):
      date  active_addresses  transfer_count  log_active_addresses  log_transfer_count  cooks_d  above_threshold
2022-01-16             21617          713598                 9.981              13.478   0.0044             True
2021-09-26             20764          645059                 9.941              13.377   0.0043             True
2022-01-09             23142          695614                10.049              13.453   0.0039             True
2021-11-28             25712          805305                10.155              13.599   0.0039             True
2021-11-22             29984          961345                10.308              13.776   0.0037             True
2021-12-25             24379          686921                10.101              13.440   0.0036             True
2021-11-07             31861         1010674                10.369              13.826   0.0035             True
2021-12-26             26292          7

### §1 Figure — Metcalfe scatter

Log-log scatter of transfer count vs active addresses per asset,
with OLS fit lines overlaid. Annotations show β estimate and
95% CI per asset. Saved to
`outputs/figures/fig_h1_metcalfe_scatter.png` at 300 DPI.

In [14]:
# §1 Figure — Metcalfe scatter
fig, ax = plt.subplots(figsize=(8, 8))

colors = {"USDC": PALETTE["primary"], "USDT": PALETTE["secondary"]}
for asset in ["USDC", "USDT"]:
    sub = h1[h1["asset"] == asset].sort_values("date")
    ax.scatter(
        sub["log_active_addresses"],
        sub["log_transfer_count"],
        s=4, alpha=0.35, color=colors[asset],
        label=None,
    )

    # OLS fit line
    res = ols_results[(asset, "levels")]
    x_range = np.linspace(
        sub["log_active_addresses"].min(),
        sub["log_active_addresses"].max(),
        100,
    )
    y_fit = res.params[0] + res.params[1] * x_range
    ax.plot(x_range, y_fit, color=colors[asset], linewidth=2,
            label=(f"{asset}: β = {res.params[1]:.3f} "
                   f"(95% CI: {res.conf_int(alpha=0.05)[1][0]:.3f}, "
                   f"{res.conf_int(alpha=0.05)[1][1]:.3f})"))

ax.set_xlabel("log(Active Addresses)", fontsize=12)
ax.set_ylabel("log(Transfer Count)", fontsize=12)
ax.set_title("H1 — Metcalfe's Law: Transfer count vs active "
             "addresses (log-log, 2020–2025)", fontsize=13)
ax.legend(loc="lower right", frameon=False, fontsize=10)

plt.tight_layout()
fig_path = FIG_DIR / "fig_h1_metcalfe_scatter.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.close(fig)

assert fig_path.exists()
print(f"Figure saved: {fig_path}")
print(f"Size: {fig_path.stat().st_size:,} bytes")

Figure saved: C:\dev\ine\outputs\figures\fig_h1_metcalfe_scatter.png
Size: 545,532 bytes


## §2 — H3 Market Concentration

Monthly HHI across stablecoins. Primary spec: OLS trend on
`time_index` with Newey-West HAC SE. Headline split at Dec 2022
(per D-01) with Jun 2022 robustness. `hhi_top5` as secondary
robustness.

**Decisions invoked:** D-01 (post-crisis cutoff),
D-05 (figure style), D-07 (output naming), D-09 (HAC maxlags = 4
for monthly data).

**Subsections (to be implemented in Prompt 4):**
- §2.1 HHI time series figure with event annotations
- §2.2 Structural event table
- §2.3 OLS trend, full window
- §2.4 OLS trend, post-Dec-2022 split (headline)
- §2.5 OLS trend, post-Jun-2022 split (robustness)
- §2.6 hhi_top5 robustness

## §3 — H4 Cost Friction

Monthly paired differences between legacy remittance cost and
on-chain fee, per rail (ETH, Tron) and per transfer size ($200,
$10,000). Test implemented as OLS of monthly differences on a
constant with HAC SE (D-10). Tron side uses `tron_median_fee_usd`
(D-02). Headline uses `legacy_flat_fee = 0`; $3.50 sensitivity
row reported underneath.

**Decisions invoked:** D-02 (paired-test design),
D-05 (figure style), D-07 (output naming), D-09 (HAC maxlags = 4),
D-10 (test implementation).

**Subsections (to be implemented in Prompt 5):**
- §3.1 Monthly fee time series figure (ETH, Tron, legacy)
- §3.2 Cost comparison bar chart at $200 and $10,000
- §3.3 Paired-test table: 4 rails/sizes × 2 flat-fee scenarios
- §3.4 ETH-Tron crossover annotation (post-Dencun 2025 months)

## §4 — H2 Diffusion & Institutional Gaps

Country-year panel of Chainalysis adoption index on macro
controls and financial-inclusion baseline. Five-spec ladder per
D-04: pooled OLS → country FE → two-way FE with
`baseline × post_2022` interaction → two-way FE excluding
forward-filled rows → two-way FE with
`baseline_year == 2024` interaction. Country-clustered SE
throughout. Specs 3–5 exclude 8 single-year countries.

**Decisions invoked:** D-04 (specification ladder),
D-05 (figure style), D-07 (output naming), D-08 (hygiene).

**Subsections (to be implemented in Prompt 6):**
- §4.1 Descriptive: adoption distribution by year/region
- §4.2 Pooled OLS with country-clustered SE
- §4.3 Country FE
- §4.4 Two-way FE with baseline × post_2022 interaction
- §4.5 Robustness: exclude forward-filled 2025 rows
- §4.6 Robustness: interact baseline with
  baseline_year == 2024

## §5 — Phase 3 Exit Checklist

To be filled in at Phase 3 close. Mirrors the Phase 2B exit
criteria pattern. Items include:
- [ ] All four hypotheses have completed analysis sections
- [ ] All figures saved to `outputs/figures/` at 300 DPI
- [ ] All tables saved to `outputs/tables/` as CSV + LaTeX
- [ ] Notebook runs top-to-bottom on a fresh kernel
- [ ] Every methodology decision logged in
  `docs/PHASE_3_DECISIONS.md`
- [ ] Global random seed set; no non-deterministic outputs
- [ ] No hardcoded paths; all paths derived from `REPO_ROOT`
- [ ] All assertions pass